# Bronze Layer

In [ ]:
CREATE OR REPLACE TABLE first_warehouse.bronze_layer.emissions_data_2023 (
    facility_id                          NUMBER,
    frs_id                               VARCHAR,
    facility_name                        VARCHAR,
    city                                 VARCHAR,
    state                                VARCHAR,
    zip_code                             VARCHAR,
    address                              VARCHAR,
    county                               VARCHAR,
    latitude                             FLOAT,
    longitude                            FLOAT,
    primary_naics_code                   NUMBER,
    industry_type_subparts               VARCHAR,
    industry_type_sectors                VARCHAR,
    total_reported_direct_emissions      FLOAT,
    co2_emissions_non_biogenic           FLOAT,
    methane_ch4_emissions                FLOAT,
    nitrous_oxide_n2o_emissions          FLOAT,
    hfc_emissions                        FLOAT,
    pfc_emissions                        FLOAT,
    sf6_emissions                        FLOAT,
    nf3_emissions                        FLOAT,
    other_fully_fluorinated_ghg_emissions FLOAT,
    hfe_emissions                        FLOAT,
    very_short_lived_compounds_emissions FLOAT,
    other_ghgs_metric_tons_co2e          FLOAT,
    biogenic_co2_emissions_metric_tons   FLOAT,
    stationary_combustion                FLOAT
);

In [ ]:
CREATE OR REPLACE FILE FORMAT first_warehouse.bronze_layer.my_csv_format
TYPE = 'CSV'
FIELD_DELIMITER = ','
SKIP_HEADER = 1  -- This skips the 1 header row
FIELD_OPTIONALLY_ENCLOSED_BY = '"'
NULL_IF = ('', 'NULL', 'confidential');


In [ ]:
SELECT $1, $2, $3
FROM @first_warehouse.bronze_layer.emissions_data_csv/emissions_data_2023.csv
(FILE_FORMAT => 'first_warehouse.bronze_layer.my_csv_format');

In [ ]:
COPY INTO first_warehouse.bronze_layer.emissions_data_2023
FROM @first_warehouse.bronze_layer.emissions_data_csv/emissions_data_2023.csv
FILE_FORMAT = (FORMAT_NAME = 'first_warehouse.bronze_layer.my_csv_format')
ON_ERROR = 'CONTINUE';

# Silver Layer

In [ ]:
CREATE OR REPLACE TABLE first_warehouse.silver_layer.emissions_data_2023 AS
    SELECT * FROM first_warehouse.bronze_layer.emissions_data_2023;

In [ ]:
DELETE FROM first_warehouse.silver_layer.emissions_data_2023 WHERE frs_id IS NULL;


In [ ]:
ALTER TABLE first_warehouse.silver_layer.emissions_data_2023 ADD COLUMN FRS_ID_new NUMBER;

UPDATE first_warehouse.silver_layer.emissions_data_2023 
SET FRS_ID_new = TRY_CAST(FRS_ID AS NUMBER);

ALTER TABLE first_warehouse.silver_layer.emissions_data_2023 DROP COLUMN FRS_ID;

ALTER TABLE first_warehouse.silver_layer.emissions_data_2023 RENAME COLUMN FRS_ID_new TO FRS_ID;

In [ ]:
TRUNCATE first_warehouse.silver_layer.emissions_data_2023;

# Gold Layer

In [ ]:
CREATE OR REPLACE TABLE first_warehouse.gold_layer.dim_facilities (
    facility_id                          NUMBER,
    frs_id                               NUMBER,
    facility_name                        VARCHAR,
    address                              VARCHAR,
    city                                 VARCHAR,
    state                                VARCHAR,
    zip_code                             VARCHAR,
    county                               VARCHAR,
    latitude                             FLOAT,
    longitude                            FLOAT,
    facility_id_int                     NUMBER);

In [ ]:
INSERT INTO first_warehouse.gold_layer.dim_facilities 
    SELECT facility_id, frs_id, facility_name, address, city, state, zip_code, county, latitude, longitude,
        row_number() OVER(ORDER BY facility_id) AS facility_id_int
    FROM first_warehouse.silver_layer.emissions_data_2023

In [ ]:
CREATE OR REPLACE TABLE first_warehouse.gold_layer.dim_industries AS
    SELECT PRIMARY_NAICS_CODE, INDUSTRY_TYPE_SUBPARTS, INDUSTRY_TYPE_SECTORS, ROW_NUMBER() OVER(ORDER BY PRIMARY_NAICS_CODE) AS industry_type_id
    FROM(
       SELECT DISTINCT PRIMARY_NAICS_CODE, INDUSTRY_TYPE_SUBPARTS, INDUSTRY_TYPE_SECTORS
        FROM first_warehouse.silver_layer.emissions_data_2023)

In [ ]:
%%sql -r dataframe_22
CREATE OR REPLACE TABLE first_warehouse.gold_layer.fact_emissions AS
    SELECT 
        gl_f.facility_id_int AS facility_id_PK, -- This is facilities' Surrogate Key
        gl_i.industry_type_id AS industry_type_PK, -- This is industries' Surrogate Key
        sl.total_reported_direct_emissions,
        sl.co2_emissions_non_biogenic,
        sl.methane_ch4_emissions,
        sl.nitrous_oxide_n2o_emissions,
        sl.hfc_emissions,
        sl.pfc_emissions,
        sl.sf6_emissions,
        sl.nf3_emissions
    FROM first_warehouse.silver_layer.emissions_data_2023 sl
    JOIN first_warehouse.gold_layer.dim_facilities gl_f ON sl.facility_id = gl_f.facility_id
    JOIN first_warehouse.gold_layer.dim_industries gl_i 
        ON  sl.primary_naics_code     = gl_i.primary_naics_code
        AND sl.industry_type_subparts = gl_i.industry_type_subparts
        AND sl.industry_type_sectors  = gl_i.industry_type_sectors;